# 不同的输入与输出方向

输入与输出允许采用不同的正交基。先算指定方向，再核对 NumPy 的分解。

依赖：NumPy、Matplotlib。按顺序运行，先在纸上预测再核对；图内使用英文标签。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 两支输入基走向哪里？

输入 v1=(1,1)/sqrt2、v2=(-1,1)/sqrt2，先手算输出。


In [ ]:
A = np.array([[2.,2.],[-1.,1.]])/np.sqrt(2)
Vmanual = np.array([[1.,-1.],[1.,1.]])/np.sqrt(2)
print('两支输入基的输出:\n',A @ Vmanual)
assert np.allclose(A @ Vmanual,np.diag([2.,1.]))


## 单位圆与两个方向

先预测哪些方向变长，再看输出椭圆。


In [ ]:
angle = np.linspace(0,2*np.pi,301)
circle = np.vstack((np.cos(angle),np.sin(angle)))
fig,axes = plt.subplots(1,2,figsize=(10,4.5),dpi=150)
for ax,M,title in zip(axes,[np.eye(2),A],['Input directions','Output directions']):
    curve = M @ circle
    ax.plot(*curve,color='#359969',lw=2)
    for j,color in enumerate(['#2878b5','#d97922']):
        vector = M @ Vmanual[:,j]
        ax.annotate('',xy=vector,xytext=(0,0),arrowprops=dict(arrowstyle='->',color=color,lw=2.5))
        ax.annotate(f'direction {j+1}',xy=vector,xytext=(-6,-18) if j == 0 else (6,6),textcoords='offset points',ha='right' if j == 0 else 'left',color=color)
    ax.set(xlim=(-2.8,2.8),ylim=(-2.8,2.8),aspect='equal',xlabel='coordinate 1',ylabel='coordinate 2',title=title)
    ax.grid(alpha=.2)
fig.tight_layout()
plt.show()


## 检查一般矩阵的分解

库返回 V^T，不是 V。特征方向的符号并不唯一，验证乘积与正交关系。


In [ ]:
for matrix in [A,np.array([[1.,1.],[0.,1.]]),
               np.array([[1.,0.],[0.,1.],[1.,1.]])]:
    U,s,Vt = np.linalg.svd(matrix,full_matrices=False)
    print('shape:',matrix.shape,'singular values:',s)
    assert np.allclose((U*s) @ Vt,matrix)
    assert np.allclose(U.T @ U,np.eye(len(s)))
    assert np.allclose(Vt @ Vt.T,np.eye(len(s)))


## 最短解与零空间

两个输入相加得到一个输出，目标是 4。有哪些解？伪逆会选哪一个？


In [ ]:
H = np.array([[1.,1.]])
b = np.array([4.])
shortest = np.linalg.pinv(H) @ b
print('最短解:',shortest)
z = np.array([1.,-1.])
for t in [-2.,0.,2.]:
    candidate = shortest+t*z
    print('候选:',candidate,'输出:',H @ candidate,'长度:',np.linalg.norm(candidate))
    assert np.allclose(H @ candidate,b)
    assert np.linalg.norm(candidate) >= np.linalg.norm(shortest)-1e-12


## 合上实验，自己解释

指出已知量、未知量、单位与计算目的。用纸上推导解释观察结果，再改变一个参数检查；有限次计算不是一般结论的证明。
